# Invoice Automation AI Agent

## AI Legends 2026 — AI agents automation

Энэ notebook нь invoice зураг болон PDF файлуудаас мэдээлэл олборлож, master database-тэй тулган шалгаж, зөрчил илрүүлж, эцсийн бизнес шийдвэр гаргах AI agent pipeline юм.

### Final decisions

- `AUTO_POST` — шууд бүртгэх боломжтой
- `HUMAN_APPROVAL` — хүний зөвшөөрөл шаардлагатай
- `DENY` — татгалзах шаардлагатай

### Required risk flags

- `AMOUNT_MISMATCH`
- `UNREGISTERED_VENDOR`
- `INVALID_DATE`
- `BANK_ACCOUNT_MISMATCH`
- `DUPLICATE`

## 1. Competition Requirement Mapping

| Competition requirement | Notebook section |
|---|---|
| Invoice image/PDF extraction | Vision extraction + PDF conversion |
| Classification | Category classification |
| Error/risk detection | Validation + risk flagging |
| Final decision | Business decision logic |
| Aggregate Q&A | Summary Q&A section |
| Reproducible output | CSV export section |

In [ ]:
# 2. Install and Import Libraries

!pip install -q pandas pymupdf pillow groq python-dateutil

import os
import re
import json
import base64
import sqlite3
import difflib
from datetime import date
from pathlib import Path

import pandas as pd
import fitz
from dateutil import parser as date_parser

from groq import Groq
from kaggle_secrets import UserSecretsClient

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

print("Libraries loaded successfully")

In [ ]:
# 3. Configuration

BASE_PATH = "/kaggle/input/competitions/ai-legends-2026-ai-agents-automation"
WORKING_PATH = "/kaggle/working"
DB_PATH = f"{BASE_PATH}/master_invoices_database.db"

SECRET_NAMES = ["GROQ_API_KEY", "API"]
MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"

print("Base path:", BASE_PATH)
print("Database path:", DB_PATH)

In [ ]:
# 4. Load Dataset and Master Database

files = os.listdir(BASE_PATH)
invoice_files = [
    os.path.join(BASE_PATH, f)
    for f in files
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".pdf"))
]

print("Total files in dataset:", len(files))
print("Invoice files:", len(invoice_files))
print("First files:", files[:10])

conn = sqlite3.connect(DB_PATH)

tables_df = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

print("Database tables:")
display(tables_df)

def read_table_if_exists(table_name):
    available = set(tables_df["name"].tolist())
    if table_name in available:
        return pd.read_sql(f"SELECT * FROM {table_name}", conn)
    return pd.DataFrame()

vendors_df = read_table_if_exists("Vendors")
items_df = read_table_if_exists("Items")
categories_df = read_table_if_exists("InvoiceCategories")
invoices_df = read_table_if_exists("Invoices")
invoice_lines_df = read_table_if_exists("InvoiceLines")

print("Vendors:", vendors_df.shape)
print("Items:", items_df.shape)
print("Categories:", categories_df.shape)
print("Historical invoices:", invoices_df.shape)

display(vendors_df.head())
display(items_df.head())
display(categories_df.head())
display(invoices_df.head())

## 5. Helper Functions

Энэ хэсэгт text, number, date, fuzzy matching, database column detection зэрэг reusable functions байна.

In [ ]:
# 5. Helper Functions

def first_existing_column(df, candidates):
    if df is None or df.empty:
        return None
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def normalize_text(value):
    if value is None or pd.isna(value):
        return ""
    text = str(value).lower().strip()
    text = text.replace("ё", "е")
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_number(value):
    if value is None or pd.isna(value):
        return None
    text = str(value)
    text = text.replace(",", "").replace("₮", "").replace("mnt", "").replace("MNT", "")
    text = re.sub(r"[^0-9.\-]", "", text)
    if text in ["", "-", "."]:
        return None
    try:
        return float(text) if "." in text else int(text)
    except Exception:
        return None

def parse_date_safe(value):
    if value is None or pd.isna(value) or str(value).strip() == "":
        return None
    try:
        return date_parser.parse(str(value), dayfirst=False, fuzzy=True).date()
    except Exception:
        return None

def fuzzy_match(query, choices, cutoff=0.55):
    if not query:
        return ""
    choices = [c for c in choices if c is not None and str(c).strip() != ""]
    if not choices:
        return query
    query_norm = normalize_text(query)
    choice_map = {normalize_text(c): c for c in choices}
    matches = difflib.get_close_matches(query_norm, list(choice_map.keys()), n=1, cutoff=cutoff)
    return choice_map[matches[0]] if matches else query

vendor_name_col = first_existing_column(vendors_df, ["vendor_name", "VendorName", "Name", "CompanyName"])
vendor_account_col = first_existing_column(vendors_df, ["account_number", "AccountNumber", "BankAccount", "bank_account"])
item_name_col = first_existing_column(items_df, ["item_description", "ItemName", "Name", "Description"])
category_name_col = first_existing_column(categories_df, ["category", "Category", "CategoryName", "Name"])
historical_invoice_number_col = first_existing_column(invoices_df, ["invoice_number", "InvoiceNumber", "number"])

print("Detected columns:")
print("vendor_name_col:", vendor_name_col)
print("vendor_account_col:", vendor_account_col)
print("item_name_col:", item_name_col)
print("category_name_col:", category_name_col)
print("historical_invoice_number_col:", historical_invoice_number_col)

## 6. Connect Groq Vision API

API key-г notebook дээр шууд бичихгүй. Kaggle Secrets ашиглана.

In [ ]:
# 6. Connect Groq Vision API

user_secrets = UserSecretsClient()

api_key = None
for secret_name in SECRET_NAMES:
    try:
        api_key = user_secrets.get_secret(secret_name)
        if api_key:
            print(f"Loaded API key from Kaggle Secret: {secret_name}")
            break
    except Exception:
        pass

if not api_key:
    raise ValueError("Groq API key not found. Add it in Kaggle Secrets as GROQ_API_KEY.")

client = Groq(api_key=api_key)
print("Groq Vision API connected")

## 7. Vision-based Invoice Extraction

Энэ хэсэг invoice image-ээс structured JSON гаргана. Prompt нь зөвхөн шаардлагатай field-үүдийг буцаах байдлаар хязгаарласан.

In [ ]:
# 7. Vision-based Invoice Extraction

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode()

EXTRACTION_PROMPT = '''
You are an invoice extraction agent.

Extract information from this invoice image and return ONLY valid JSON.
Do not use markdown. Do not add extra comments.

Required JSON keys:
{
  "invoice_number": "",
  "invoice_date": "",
  "due_date": "",
  "vendor_name": "",
  "bank_name": "",
  "account_number": "",
  "email": "",
  "item_description": "",
  "quantity": "",
  "unit_price": "",
  "subtotal": "",
  "tax": "",
  "total_amount": "",
  "currency": ""
}

Rules:
- If a value is unclear, return an empty string.
- Keep invoice_number exactly as shown.
- Extract only one main item description if multiple items exist.
- Use numeric strings for quantity, unit_price, subtotal, tax, and total_amount.
'''

def extract_invoice_from_image(image_path):
    base64_image = encode_image(image_path)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": EXTRACTION_PROMPT},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        temperature=0
    )

    raw_text = response.choices[0].message.content.strip()
    raw_text = raw_text.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(raw_text)
    except Exception as e:
        return {
            "raw_response": raw_text,
            "extraction_error": str(e)
        }

## 8. PDF and Image Processing

PDF invoice-ийг эхний page image болгож хөрвүүлээд extraction function-д оруулна.

In [ ]:
# 8. PDF and Image Processing

def convert_pdf_to_image(pdf_path):
    pdf_name = Path(pdf_path).stem
    output_image_path = f"{WORKING_PATH}/{pdf_name}_page1.png"

    doc = fitz.open(pdf_path)
    page = doc[0]
    pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
    pix.save(output_image_path)
    doc.close()

    return output_image_path

def prepare_file_for_extraction(file_path):
    lower = file_path.lower()
    if lower.endswith((".jpg", ".jpeg", ".png")):
        return file_path
    if lower.endswith(".pdf"):
        return convert_pdf_to_image(file_path)
    return None

## 9. Data Normalization and Category Classification

Энэ хэсэг extraction output-ийг тооцоолол хийхэд тохиромжтой structured хэлбэрт оруулна.

In [ ]:
# 9. Data Normalization and Category Classification

def get_vendor_choices():
    if vendor_name_col and not vendors_df.empty:
        return vendors_df[vendor_name_col].dropna().astype(str).tolist()
    return []

def is_vendor_registered(vendor_name):
    vendor_choices = get_vendor_choices()
    if not vendor_choices:
        return False
    matched_vendor = fuzzy_match(vendor_name, vendor_choices, cutoff=0.60)
    return normalize_text(matched_vendor) in [normalize_text(v) for v in vendor_choices]

def get_matched_vendor_name(vendor_name):
    vendor_choices = get_vendor_choices()
    if not vendor_choices:
        return vendor_name
    return fuzzy_match(vendor_name, vendor_choices, cutoff=0.60)

def classify_category(item_description, vendor_name=""):
    if category_name_col and not categories_df.empty:
        categories = categories_df[category_name_col].dropna().astype(str).tolist()
        text = f"{item_description} {vendor_name}"
        return fuzzy_match(text, categories, cutoff=0.25)

    text = normalize_text(f"{item_description} {vendor_name}")
    rules = {
        "Office Supplies": ["paper", "pen", "printer", "office", "stationery", "бичиг", "цаас"],
        "Utilities": ["electric", "water", "internet", "utility", "цахилгаан", "ус", "интернет"],
        "Travel Expense": ["hotel", "flight", "taxi", "travel", "шатахуун", "унаа", "зочид"],
        "Software": ["software", "subscription", "license", "app", "cloud"],
        "Maintenance": ["repair", "maintenance", "service", "засвар", "үйлчилгээ"],
    }

    for category, keywords in rules.items():
        if any(k in text for k in keywords):
            return category

    return "Other"

def normalize_invoice_data(raw_data, source_file):
    vendor_raw = raw_data.get("vendor_name", "")
    matched_vendor = get_matched_vendor_name(vendor_raw)

    invoice_date_parsed = parse_date_safe(raw_data.get("invoice_date"))
    due_date_parsed = parse_date_safe(raw_data.get("due_date"))

    item_description = raw_data.get("item_description", "")
    category = classify_category(item_description, matched_vendor)

    return {
        "source_file": source_file,
        "invoice_number": raw_data.get("invoice_number", ""),
        "invoice_date": raw_data.get("invoice_date", ""),
        "due_date": raw_data.get("due_date", ""),
        "invoice_date_parsed": invoice_date_parsed,
        "due_date_parsed": due_date_parsed,
        "vendor_name_raw": vendor_raw,
        "vendor_name": matched_vendor,
        "bank_name": raw_data.get("bank_name", ""),
        "account_number": str(raw_data.get("account_number", "")).strip(),
        "email": raw_data.get("email", ""),
        "item_description": item_description,
        "quantity": normalize_number(raw_data.get("quantity")),
        "unit_price": normalize_number(raw_data.get("unit_price")),
        "subtotal": normalize_number(raw_data.get("subtotal")),
        "tax": normalize_number(raw_data.get("tax")),
        "total_amount": normalize_number(raw_data.get("total_amount")),
        "currency": raw_data.get("currency", ""),
        "category": category,
        "raw_extraction": json.dumps(raw_data, ensure_ascii=False),
        "processing_status": "SUCCESS" if "extraction_error" not in raw_data else "EXTRACTION_ERROR"
    }

## 10. Validation Rules and Risk Flagging

Энэ хэсэг competition-д шаардсан зөрчлүүдийг илрүүлнэ.

In [ ]:
# 10. Validation Rules and Risk Flagging

def check_amount_mismatch(row):
    quantity = row.get("quantity")
    unit_price = row.get("unit_price")
    total = row.get("total_amount")

    if quantity is None or unit_price is None or total is None:
        return False

    calculated = quantity * unit_price
    tolerance = max(1, abs(total) * 0.01)
    return abs(calculated - total) > tolerance

def check_invalid_date(row):
    invoice_date = row.get("invoice_date_parsed")
    due_date = row.get("due_date_parsed")

    if invoice_date is None:
        return True

    today = date.today()
    if invoice_date > today:
        return True

    if due_date is not None and due_date < invoice_date:
        return True

    return False

def check_bank_account_mismatch(row):
    if not vendor_account_col or not vendor_name_col or vendors_df.empty:
        return False

    account = normalize_text(row.get("account_number", ""))
    vendor = normalize_text(row.get("vendor_name", ""))

    if not account:
        return False

    possible = vendors_df.copy()
    possible["_vendor_norm"] = possible[vendor_name_col].astype(str).map(normalize_text)

    vendor_rows = possible[possible["_vendor_norm"] == vendor]
    if vendor_rows.empty:
        return False

    valid_accounts = vendor_rows[vendor_account_col].dropna().astype(str).map(normalize_text).tolist()
    if not valid_accounts:
        return False

    return account not in valid_accounts

def check_duplicate(row):
    if invoices_df.empty or not historical_invoice_number_col:
        return False

    invoice_number = normalize_text(row.get("invoice_number", ""))
    if not invoice_number:
        return False

    historical_numbers = invoices_df[historical_invoice_number_col].dropna().astype(str).map(normalize_text).tolist()
    return invoice_number in historical_numbers

def build_risk_flags(row):
    flags = []

    required_fields = ["invoice_number", "invoice_date", "vendor_name", "total_amount"]
    missing_required = any(row.get(f) in [None, ""] for f in required_fields)

    if missing_required:
        flags.append("MISSING_REQUIRED_FIELD")

    if not is_vendor_registered(row.get("vendor_name", "")):
        flags.append("UNREGISTERED_VENDOR")

    if check_amount_mismatch(row):
        flags.append("AMOUNT_MISMATCH")

    if check_invalid_date(row):
        flags.append("INVALID_DATE")

    if check_bank_account_mismatch(row):
        flags.append("BANK_ACCOUNT_MISMATCH")

    if check_duplicate(row):
        flags.append("DUPLICATE")

    if row.get("processing_status") != "SUCCESS":
        flags.append("LOW_CONFIDENCE_EXTRACTION")

    return flags

def decide_final_action(risk_flags):
    serious_deny_flags = {
        "AMOUNT_MISMATCH",
        "INVALID_DATE",
        "BANK_ACCOUNT_MISMATCH",
        "DUPLICATE"
    }

    review_flags = {
        "UNREGISTERED_VENDOR",
        "MISSING_REQUIRED_FIELD",
        "LOW_CONFIDENCE_EXTRACTION"
    }

    flag_set = set(risk_flags)

    if flag_set & serious_deny_flags:
        return "DENY"

    if flag_set & review_flags:
        return "HUMAN_APPROVAL"

    return "AUTO_POST"

def explain_decision(risk_flags, final_decision):
    if not risk_flags:
        return "No risk detected. Invoice can be posted automatically."
    return f"Decision is {final_decision} because these risk flags were detected: {', '.join(risk_flags)}."

def validate_invoice(row):
    risk_flags = build_risk_flags(row)
    final_decision = decide_final_action(risk_flags)

    row = dict(row)
    row["amount_check"] = "FAIL" if "AMOUNT_MISMATCH" in risk_flags else "PASS"
    row["date_check"] = "FAIL" if "INVALID_DATE" in risk_flags else "PASS"
    row["bank_account_check"] = "FAIL" if "BANK_ACCOUNT_MISMATCH" in risk_flags else "PASS"
    row["duplicate_check"] = "FAIL" if "DUPLICATE" in risk_flags else "PASS"
    row["is_registered_vendor"] = "UNREGISTERED_VENDOR" not in risk_flags
    row["risk_flags"] = risk_flags
    row["risk_flags_text"] = ", ".join(risk_flags) if risk_flags else "NONE"
    row["final_decision"] = final_decision
    row["explanation"] = explain_decision(risk_flags, final_decision)

    return row

## 11. Batch Processing

Demo үед эхний хэдэн файл боловсруулж болно. Эцсийн submission хийхдээ `MAX_FILES = None` болгож бүх invoice-ийг ажиллуулна.

In [ ]:
# 11. Batch Processing

MAX_FILES = None  # Demo хийх бол 5 гэж тавьж болно. Final run дээр None байлгана.

files_to_process = invoice_files if MAX_FILES is None else invoice_files[:MAX_FILES]

all_results = []
failed_files = []

print("Files to process:", len(files_to_process))

for idx, file_path in enumerate(files_to_process, start=1):
    source_file = os.path.basename(file_path)
    print(f"[{idx}/{len(files_to_process)}] Processing: {source_file}")

    try:
        extraction_path = prepare_file_for_extraction(file_path)

        if extraction_path is None:
            failed_files.append({
                "source_file": source_file,
                "error": "Unsupported file type"
            })
            continue

        raw_data = extract_invoice_from_image(extraction_path)
        normalized = normalize_invoice_data(raw_data, source_file)
        validated = validate_invoice(normalized)
        all_results.append(validated)

    except Exception as e:
        failed_files.append({
            "source_file": source_file,
            "error": str(e)
        })

final_df = pd.DataFrame(all_results)
failed_df = pd.DataFrame(failed_files)

print("Processed successfully:", len(final_df))
print("Failed:", len(failed_df))

display(final_df.head())
display(failed_df.head())

## 12. Results Review

Энэ хэсэг clean, suspicious, denied invoice-уудыг ялгаж харуулна.

In [ ]:
# 12. Results Review

if not final_df.empty:
    suspicious_df = final_df[final_df["risk_flags_text"] != "NONE"].copy()
    clean_df = final_df[final_df["risk_flags_text"] == "NONE"].copy()
    denied_df = final_df[final_df["final_decision"] == "DENY"].copy()
    human_approval_df = final_df[final_df["final_decision"] == "HUMAN_APPROVAL"].copy()
    auto_post_df = final_df[final_df["final_decision"] == "AUTO_POST"].copy()
else:
    suspicious_df = clean_df = denied_df = human_approval_df = auto_post_df = pd.DataFrame()

print("========== RESULT SUMMARY ==========")
print("Total processed invoices :", len(final_df))
print("AUTO_POST                :", len(auto_post_df))
print("HUMAN_APPROVAL           :", len(human_approval_df))
print("DENY                     :", len(denied_df))
print("Suspicious invoices      :", len(suspicious_df))
print("Clean invoices           :", len(clean_df))
print("Failed files             :", len(failed_df))
print("====================================")

important_cols = [
    "source_file",
    "invoice_number",
    "vendor_name",
    "total_amount",
    "category",
    "risk_flags_text",
    "final_decision",
    "explanation"
]

display(final_df[[c for c in important_cols if c in final_df.columns]].head(20))

## 13. Aggregate Q&A

Competition-ийн шаардсан нийт үр дүн дээр reasoning хийх хэсэг.

In [ ]:
# 13. Aggregate Q&A

def count_flag(df, flag):
    if df.empty or "risk_flags" not in df.columns:
        return 0
    return df["risk_flags"].apply(lambda flags: flag in flags if isinstance(flags, list) else flag in str(flags)).sum()

qa_answers = {
    "Хэдэн invoice AUTO_POST болсон бэ?": len(auto_post_df),
    "Хэдэн invoice HUMAN_APPROVAL болсон бэ?": len(human_approval_df),
    "Хэдэн invoice DENY болсон бэ?": len(denied_df),
    "Хэдэн invoice duplicate байсан бэ?": count_flag(final_df, "DUPLICATE"),
    "Хэдэн invoice бүртгэлгүй vendor-той байсан бэ?": count_flag(final_df, "UNREGISTERED_VENDOR"),
    "Хэдэн invoice amount mismatch-тэй байсан бэ?": count_flag(final_df, "AMOUNT_MISMATCH"),
    "Хэдэн invoice invalid date-тэй байсан бэ?": count_flag(final_df, "INVALID_DATE"),
    "Хэдэн invoice bank account mismatch-тэй байсан бэ?": count_flag(final_df, "BANK_ACCOUNT_MISMATCH"),
}

for question, answer in qa_answers.items():
    print(f"Q: {question}")
    print(f"A: {answer}")
    print("-" * 60)

if not final_df.empty and "category" in final_df.columns:
    print("Category distribution:")
    category_summary = final_df["category"].value_counts().reset_index()
    category_summary.columns = ["category", "count"]
    display(category_summary)

## 14. Export Final Results

CSV файлуудыг Kaggle `/working` folder руу хадгална.

In [ ]:
# 14. Export Final Results

final_output_path = f"{WORKING_PATH}/final_invoice_results.csv"
suspicious_output_path = f"{WORKING_PATH}/suspicious_invoice_results.csv"
clean_output_path = f"{WORKING_PATH}/clean_invoice_results.csv"
failed_output_path = f"{WORKING_PATH}/failed_invoice_files.csv"

export_final_df = final_df.copy()
if "risk_flags" in export_final_df.columns:
    export_final_df["risk_flags"] = export_final_df["risk_flags"].apply(lambda x: json.dumps(x, ensure_ascii=False))

export_final_df.to_csv(final_output_path, index=False, encoding="utf-8-sig")
suspicious_df.to_csv(suspicious_output_path, index=False, encoding="utf-8-sig")
clean_df.to_csv(clean_output_path, index=False, encoding="utf-8-sig")
failed_df.to_csv(failed_output_path, index=False, encoding="utf-8-sig")

print("Saved:", final_output_path)
print("Saved:", suspicious_output_path)
print("Saved:", clean_output_path)
print("Saved:", failed_output_path)

## 15. Project Summary

Энэ хэсэг видео болон writeup-д ашиглаж болох товч summary гаргана.

In [ ]:
# 15. Project Summary

print("========== ТӨСЛИЙН ХУРААНГУЙ ==========")
print("Төслийн нэр: Invoice Automation AI Agent")
print("Pipeline: PDF/JPG/PNG → Vision Extraction → DB Validation → Risk Flagging → Final Decision → Q&A")
print(f"Нийт боловсруулсан invoice     : {len(final_df)}")
print(f"AUTO_POST                      : {len(auto_post_df)}")
print(f"HUMAN_APPROVAL                 : {len(human_approval_df)}")
print(f"DENY                           : {len(denied_df)}")
print(f"Сэжигтэй invoice               : {len(suspicious_df)}")
print(f"Алдаа гарсан файлууд           : {len(failed_df)}")
print("Гол risk labels: AMOUNT_MISMATCH, UNREGISTERED_VENDOR, INVALID_DATE, BANK_ACCOUNT_MISMATCH, DUPLICATE")
print("======================================")

## 16. Limitations and Future Improvements

### Limitations

- Low-resolution invoice images can reduce extraction accuracy.
- Some unusual invoice layouts may need prompt tuning.
- Duplicate detection currently depends mainly on invoice number/history.
- Category classification uses historical categories and fallback keyword logic.

### Future Improvements

- Field-level confidence scores
- Embedding-based duplicate detection
- Human approval dashboard
- Stronger category classifier
- Integration with ERP posting workflow